In [1]:
import pandas as pd

calendar = pd.read_csv('../data/calendar.csv')
print(calendar.shape)
calendar.head()

(5623190, 5)


,listing_id,date,available,minimum_nights,maximum_nights
0,18674,2026-06-25,f,1,999
1,18674,2026-06-26,f,2,999
2,18674,2026-06-27,f,2,999
3,18674,2026-06-28,t,1,999
4,18674,2026-06-29,f,3,999


In [2]:
print(calendar.shape)
print(calendar.columns.tolist())

(5623190, 5)
['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']


In [3]:
calendar['date'] = pd.to_datetime(calendar['date'])
print(calendar['date'].min(), calendar['date'].max())
print(calendar['available'].value_counts())

2026-06-24 00:00:00 2027-07-02 00:00:00
available
t    3281038
f    2342152
Name: count, dtype: int64


In [4]:
# Load the cleaned listings to get the relevant listing IDs
listings = pd.read_csv('../data/listings_for_sql.csv')

# Filter to entire apartments in client districts
relevant_ids = listings[
    (listings['is_entire_apt'] == True) &
    (listings['client_district'] == True)
]['id'].tolist()

print("Relevant listings:", len(relevant_ids))

# Filter calendar to just these listings
calendar_filtered = calendar[calendar['listing_id'].isin(relevant_ids)].copy()
print("Filtered calendar rows:", calendar_filtered.shape[0])

Relevant listings: 6957
Filtered calendar rows: 2539305


In [5]:
calendar_filtered['month'] = calendar_filtered['date'].dt.to_period('M').astype(str)

# Booking rate = % of nights NOT available (i.e., booked)
monthly_demand = calendar_filtered.groupby('month')['available'].apply(
    lambda x: (x.str.strip() == 'f').mean() * 100
).reset_index()
monthly_demand.columns = ['month', 'booked_pct']

print(monthly_demand)

      month  booked_pct
0   2026-06   76.042224
1   2026-07   61.570192
2   2026-08   40.115085
3   2026-09   32.071295
4   2026-10   27.298103
5   2026-11   18.627282
6   2026-12   17.996263
7   2027-01   27.272601
8   2027-02   29.336331
9   2027-03   31.408143
10  2027-04   39.266925
11  2027-05   41.997153
12  2027-06   47.758896
13  2027-07   47.715736


In [6]:
scrape_date = pd.Timestamp('2026-06-24')
calendar_filtered['days_out'] = (calendar_filtered['date'] - scrape_date).dt.days

# Bucket into 30-day horizon windows
calendar_filtered['horizon_bucket'] = (calendar_filtered['days_out'] // 30) * 30

calendar_filtered['is_booked'] = calendar_filtered['available'].str.strip() == 'f'

# Booking rate by horizon bucket ALONE (ignoring month) - this shows the pure horizon effect
horizon_only = calendar_filtered.groupby('horizon_bucket')['is_booked'].mean() * 100
print(horizon_only)

horizon_bucket
0      67.047940
30     45.825787
60     32.348234
90     29.575008
120    20.855254
150    16.707393
180    24.690719
210    28.923387
240    29.202242
270    37.703033
300    39.758517
330    46.546404
360    48.524460
Name: is_booked, dtype: float64


In [7]:
# Compare booking rate by month, but only within the SAME horizon bucket range
# This controls for horizon effect - if seasonality is real, some months should
# still stand out even at the same distance from the scrape date
pivot = calendar_filtered.groupby(['horizon_bucket', 'month'])['is_booked'].mean().reset_index()
pivot['booked_pct'] = pivot['is_booked'] * 100

# Focus on horizon buckets that have multiple months represented
print(pivot.sort_values(['horizon_bucket', 'month']).to_string())

    horizon_bucket    month  is_booked  booked_pct
0                0  2026-06   0.760422   76.042224
1                0  2026-07   0.647312   64.731200
2               30  2026-07   0.525047   52.504672
3               30  2026-08   0.433971   43.397102
4               60  2026-08   0.320924   32.092377
5               60  2026-09   0.324579   32.457888
6               90  2026-09   0.311692   31.169246
7               90  2026-10   0.288918   28.891764
8              120  2026-10   0.239514   23.951416
9              120  2026-11   0.193072   19.307173
10             150  2026-11   0.172675   17.267500
11             150  2026-12   0.164273   16.427339
12             180  2026-12   0.208489   20.848851
13             180  2027-01   0.269150   26.914959
14             210  2027-01   0.278389   27.838867
15             210  2027-02   0.296464   29.646399
16             240  2027-02   0.287782   28.778209
17             240  2027-03   0.294143   29.414259
18             270  2027-03   0

In [8]:
# Simple month-level summary for the dashboard (using the raw monthly view, now that we've validated it's not purely a horizon artifact)
monthly_demand.to_csv('../data/monthly_demand.csv', index=False)
print("Saved monthly_demand.csv")

Saved monthly_demand.csv
